# ECON 525: Data Management & Visualization in Economics
## Classification: learning from a heart-disease dataset

The [case-study slides](slides-ch05/05classification_casestudy.html) introduce the decisions behind a classification project. Here we carry those decisions into Python: inspect the records, define the outcome correctly, explore the training data, fit a model, and evaluate its mistakes.

Our question is whether measurements in these historical records help distinguish **recorded disease presence from absence**. This is not a prediction of a future heart attack. The small teaching dataset does not establish how a model would perform in a hospital.

By the end, you should be able to explain what a row and a target mean, separate training/validation/test data, prepare numerical and categorical features, and interpret a confusion matrix. The optional sections extend the same workflow to more classifiers, tuning, and experiment tracking.


### Tools and data

Open the notebook from the `05-appliedML` folder, with `heart.csv` beside it, and run the cells from top to bottom. We reuse pandas, NumPy, matplotlib, and scikit-learn from `01-introML.ipynb`, and `display` from the web-scraping lesson. The new plotting library is **seaborn**, which builds on matplotlib to draw statistical plots directly from DataFrames. `display` shows a table even when it is not the last line of a cell. Modeling imports appear where we need them.

### Optional setup: install or update seaborn

Run the next cell once if seaborn is missing or older than 0.13.2. The course Codespace may already have a suitable version, in which case skip this cell.

`%pip` installs into the Python environment selected for this notebook. **After the installation finishes, restart the notebook kernel, then continue from the imports cell below.** A restart is needed if Python has already loaded the old seaborn version. 

In [ ]:
%pip install "seaborn>=0.13.2"


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display

sns.set_theme(style="whitegrid")
class_colors = {0: "#2b6cb0", 1: "#d97706"}
print("Seaborn version:", sns.__version__)


## 1. Understand the records before modeling

`pd.read_csv` reads the supplied file into a DataFrame named `raw_data`. We copy it to `data` for our work, keeping the raw table available. Each row is a record with 13 measurements and one stored outcome. `shape` gives the number of rows and columns; `head()` displays the first five records.


In [ ]:
raw_data = pd.read_csv("heart.csv")
data = raw_data.copy()
print("Rows and columns:", data.shape)
data.head()


### A crucial detail: the stored target is reversed

The [UCI Heart Disease dataset](https://archive.ics.uci.edu/dataset/45/heart+disease) describes a diagnostic outcome: `num = 0` means absence of disease; values 1–4 mean presence. The supplied CSV is a recoded version of the Cleveland data. Its stored `target = 1` corresponds to **absence**, and `target = 0` to **presence**.

For this notebook we reverse that binary code, so that **`target = 1` means recorded disease presence** and **`target = 0` means absence**. Subtracting a binary value from 1 swaps 0 and 1. We call this table `labeled_data` and keep `raw_data` unchanged. The copy named `data` lets us inspect the corrected labels before cleaning.


In [ ]:
labeled_data = raw_data.copy()
labeled_data["target"] = 1 - raw_data["target"]
data = labeled_data.copy()
data["target"].value_counts().sort_index().rename("records")


### Data dictionary

The following codes describe **this CSV**, which uses different category numbers from the original UCI file. 

| Column | Meaning and units / codes |
|---|---|
| `age` | Age in years |
| `sex` | Recorded sex: 0 female, 1 male |
| `cp` | Chest pain: 0 asymptomatic, 1 atypical angina, 2 non-anginal pain, 3 typical angina |
| `trestbps` | Resting blood pressure, mm Hg |
| `chol` | Serum cholesterol, mg/dL |
| `fbs` | Fasting blood sugar >120 mg/dL: 0 false, 1 true |
| `restecg` | Resting ECG: 0 probable/definite left ventricular hypertrophy, 1 normal, 2 ST-T abnormality |
| `thalach` | Maximum heart rate achieved, beats per minute |
| `exang` | Exercise-induced angina: 0 no, 1 yes |
| `oldpeak` | Exercise-induced ST depression relative to rest |
| `slope` | Peak exercise ST slope: 0 downsloping, 1 flat, 2 upsloping |
| `ca` | Number of major vessels colored by fluoroscopy: 0–3; code 4 corresponds to missing information |
| `thal` | Thallium stress-test result: 1 fixed defect, 2 normal, 3 reversible defect; code 0 corresponds to missing information |
| `target` | **After our reversal:** 0 absence, 1 presence of recorded disease |

An integer can represent a category rather than an amount. For example, chest-pain code 3 is not three times code 1. `thal` is a test result, not a diagnosis of thalassemia.


### Missing values can be hidden in codes

`isna().sum()` counts recognized missing entries in each column. It will not recognize special codes such as `ca = 4` or `thal = 0`. We inspect those separately. `duplicated().sum()` counts rows that exactly repeat an earlier row.


In [ ]:
display(data.isna().sum().rename("recognized_missing"))
display(data[["ca", "thal"]].apply(pd.Series.value_counts))
print("Exact repeated rows:", data.duplicated().sum())


### Activity 1: Find the records that need attention


**Pairs · 4 minutes.** Count rows with `ca == 4`, rows with `thal == 0`, and rows with either code. Use `|` for “or” and put each comparison in parentheses. Deliver the three counts and one sentence explaining why an all-zero `isna()` result is not enough.

**Debrief:** Why might the sum of the first two counts differ from the count with either code?


In [ ]:
# Count the two codes, then combine the conditions with |.
# ca_count = ...
# thal_count = ...
# either_count = ...


**Your response:**

…


### Apply an explicit classroom cleaning rule

For this small demonstration we exclude the seven records with missing `ca` or `thal` information. This complete-case choice can change who remains in the sample; it is not a rule to drop missing data in every project. We also treat an exact repeated row as a duplicated record and retain its first occurrence. Without patient IDs we cannot prove that two identical rows describe the same person, so this is an explicit teaching assumption.

We apply these fixed rules before splitting so that exact copies cannot appear on both sides of our evaluation. `cleaned_data` keeps the remaining records with their original category codes and our corrected target. We do not remove a valid measurement just because it is unusually large. In a larger project, investigate its source first; if imputing values, learn the imputation rule using training data only.


In [ ]:
valid_codes = (labeled_data["ca"] < 4) & (labeled_data["thal"] > 0)
cleaned_data = labeled_data.loc[valid_codes].drop_duplicates().copy()
data = cleaned_data.copy()
print("Rows after cleaning:", len(data))


### Make column names and categories readable

`rename` changes column labels. A dictionary passed to `map` translates each code into a readable value; assigning the result to the whole column avoids chained assignment. We start from `cleaned_data`, which still has numeric category codes, and put the readable version in `data`. The mappings below are fixed definitions, not patterns learned from the data.


In [ ]:
data = cleaned_data.rename(columns={
    "cp": "chest_pain_type", "trestbps": "resting_blood_pressure",
    "chol": "cholesterol", "fbs": "fasting_blood_sugar",
    "restecg": "resting_electrocardiogram", "thalach": "max_heart_rate_achieved",
    "exang": "exercise_induced_angina", "oldpeak": "st_depression",
    "slope": "st_slope", "ca": "num_major_vessels",
    "thal": "thallium_test_result"
})

category_labels = {
    "sex": {0: "female", 1: "male"},
    "chest_pain_type": {0: "asymptomatic", 1: "atypical angina",
                        2: "non-anginal pain", 3: "typical angina"},
    "fasting_blood_sugar": {0: "at most 120 mg/dL", 1: "above 120 mg/dL"},
    "resting_electrocardiogram": {0: "ventricular hypertrophy", 1: "normal",
                                 2: "ST-T abnormality"},
    "exercise_induced_angina": {0: "no", 1: "yes"},
    "st_slope": {0: "downsloping", 1: "flat", 2: "upsloping"},
    "thallium_test_result": {1: "fixed defect", 2: "normal", 3: "reversible defect"}
}
for column, labels in category_labels.items():
    data[column] = data[column].map(labels)

display(data.head())


## 2. Keep different data for learning and evaluation

**Features** are the inputs available to the model; the **target** is the outcome we want it to predict. `X` contains all feature columns and `y` contains only the target. The target must not be an input: that would reveal the answer.

We split into approximately **60% training**, **20% validation**, and **20% test** records:

| Part | Purpose |
|---|---|
| Training | Explore patterns and fit preprocessing and model parameters |
| Validation | Compare proposed models or thresholds while developing the workflow |
| Test | Evaluate the frozen workflow once at the end |

First we reserve 20% as test data. Then we reserve 25% of the remaining 80% for validation: 0.25 × 0.80 = 0.20. `stratify` keeps class proportions similar across splits; `random_state=0` makes the split reproducible. It does not guarantee a representative or unbiased sample. In data with repeated patients, split by patient; with future predictions, consider a time-based split.


In [ ]:
from sklearn.model_selection import train_test_split

X = data.drop(columns="target")
y = data["target"]

X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=0
)
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=0
)
print("Training / validation / test rows:", len(X_train), len(X_val), len(X_test))
train_data = X_train.assign(target=y_train)


## 3. Explore the training data

Now that evaluation records are set aside, we use `train_data` for exploratory data analysis (EDA). `.assign(target=y_train)` temporarily joins the training outcomes to the training features so plots can compare the two classes. It does not add the target to `X_train`.

### Class balance and a simple benchmark

`value_counts()` counts each class. With `normalize=True`, it returns proportions. A model that always guesses the most common class can achieve that class's share as accuracy while missing every record in the other class.


In [ ]:
display(train_data["target"].value_counts().sort_index().rename("records"))
display(train_data["target"].value_counts(normalize=True).sort_index().rename("share"))
ax = sns.countplot(data=train_data, x="target", color="#2b6cb0")
ax.set(xlabel="Recorded disease (0 = absent, 1 = present)", ylabel="Training records")
plt.show()


### Numerical summaries and distributions

We treat six columns as numerical measurements or counts and seven as categories. The number of vessels is a discrete count, so we use a count plot for it. `describe().T` places one feature per row and summarizes its observed range and spread. A sample average is a description of these training records, not a clinical cutoff.


In [ ]:
num_feats = ["age", "cholesterol", "resting_blood_pressure",
             "max_heart_rate_achieved", "st_depression", "num_major_vessels"]
cat_feats = list(category_labels)
display(train_data[num_feats].describe().T.round(2))


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 11))
for column, ax in zip(num_feats, axes.flat):
    if column == "num_major_vessels":
        sns.countplot(data=train_data, x=column, hue="target", palette=class_colors, ax=ax)
    else:
        sns.histplot(data=train_data, x=column, hue="target", element="step",
                     bins=15, palette=class_colors, ax=ax)
    ax.set_title(column.replace("_", " "))
fig.suptitle("Training distributions by recorded disease class", y=1.01)
plt.tight_layout()
plt.show()


### Compare categories using the right denominator

A count plot shows how many records fall in each category. To compare the share with disease, use a row-normalized cross-tabulation. `normalize="index"` makes the values in **each row** add to one. A large count may reflect a large group rather than a high within-group disease share.

These are associations in a selected historical sample. They do not show that changing a measurement would change disease risk, or that the same relationship holds in today's patient population.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.countplot(data=train_data, x="chest_pain_type", hue="target", palette=class_colors, ax=ax)
ax.set(xlabel="Chest-pain category", ylabel="Training records")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

display(pd.crosstab(train_data["chest_pain_type"], train_data["target"],
                    normalize="index").round(2))


### Activity 2: Counts or shares?


**Pairs · 5 minutes.** Use `pd.crosstab` to compare `exercise_induced_angina` with `target`. Produce a count table and a table with `normalize="index"`. Identify which group has the larger share of recorded disease and describe the comparison in one sentence.

**Debrief:** Why does this table not prove that exercise-induced angina causes disease?


In [ ]:
# Build the count table, then the within-row share table.
# counts = pd.crosstab(...)
# shares = pd.crosstab(..., normalize="index")


**Your response:**

…


### Correlations describe linear associations

Pearson correlation ranges from −1 to +1. Values near zero indicate little **linear** association, not necessarily no relationship. This heatmap compares numerical features with each other; it does not rank their predictive value for the target. We do not turn category codes into quantities just to calculate correlations.


In [ ]:
corr = train_data[num_feats].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="vlag", vmin=-1, vmax=1,
            center=0, ax=ax)
ax.set_title("Numerical features: training-data correlations")
plt.tight_layout()
plt.show()


## 4. Turn a table into model inputs

We start with **logistic regression**, a classifier that estimates a probability for class 1 from a weighted combination of input features. Despite its name, here it predicts a category. Its fitted weights describe predictive associations, not causal effects.

Two preparation steps help this model:

- **Scale numerical features:** subtract each training-column mean and divide by its training-column standard deviation. This puts values such as age and cholesterol on comparable scales.
- **One-hot encode categories:** create indicator columns. For example, a category can become a column that is 1 when a row belongs to it and 0 otherwise. This avoids pretending that category numbers are ordered amounts.

Here is a small illustration of indicators. `get_dummies` is useful for seeing the idea; the actual model will use scikit-learn's encoder so the training and validation columns stay aligned.


In [ ]:
example = pd.DataFrame({"chest_pain": ["typical", "asymptomatic", "typical"]})
display(example)
display(pd.get_dummies(example, dtype=int))


### A pipeline keeps preparation attached to the model

`ColumnTransformer` applies different preparation to named groups of columns. `Pipeline` performs the steps in order. The names `preprocess` and `classifier` are labels we choose for these steps.

When we call `model.fit(X_train, y_train)`, it learns the scaling values and category vocabulary **from the training rows**, transforms those rows, and fits logistic regression. Later `predict` applies the same learned preparation automatically. Calling `fit` on validation or test data would let evaluation information influence training: **data leakage**.

`OneHotEncoder` returns numeric columns. `drop="first"` omits one reference category per feature; `handle_unknown="ignore"` allows a new category at prediction time rather than raising an error (its indicators become all zero, so investigate such cases). `sparse_output=False` gives a regular dense array for this small dataset. `max_iter=1000` allows up to 1,000 optimization iterations; it is not 1,000 training rows.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

preprocess = ColumnTransformer([
    ("numeric", StandardScaler(), num_feats),
    ("categorical", OneHotEncoder(drop="first", handle_unknown="ignore",
                                  sparse_output=False), cat_feats)
])
model = Pipeline([
    ("preprocess", preprocess),
    ("classifier", LogisticRegression(max_iter=1000, random_state=0))
])
model.fit(X_train, y_train)


### A probability and a predicted class are different outputs

`predict_proba(X_val)` returns one probability column for each class; `model.classes_` gives their order. With classes `[0, 1]`, `[:, 1]` selects the class-1 probability for every row. The default class prediction picks the larger probability (for binary logistic regression, class 1 when its probability exceeds 0.5; an exact tie goes to class 0).

Each row below is a validation record the model did not fit on. `actual` is its known outcome, `probability_of_disease` is the model's estimate, and `predicted` is the resulting class. These probabilities have not been established as calibrated clinical risks.


In [ ]:
print("Probability column order:", model.classes_)
val_probability = model.predict_proba(X_val)[:, 1]
val_prediction = model.predict(X_val)
display(pd.DataFrame({
    "actual": y_val, "probability_of_disease": val_probability,
    "predicted": val_prediction
}).head())


## 5. Evaluate the mistakes, not only the accuracy

We define **positive** as recorded disease presence (`target=1`). A confusion matrix counts actual classes in its rows and predicted classes in its columns:

| | Predicted 0 | Predicted 1 |
|---|---|---|
| Actual 0 | True negative (TN) | False positive (FP) |
| Actual 1 | False negative (FN) | True positive (TP) |

Accuracy is the share correct. Recall asks how many actual positive records were detected. Precision asks how many predicted positive records were actually positive. In a screening setting, missed cases and false alarms both have costs; choosing a metric or threshold requires that context.

| Metric | Calculation | Interpretation |
|---|---|---|
| Accuracy | (TP + TN) / all records | Everyone evaluated |
| Recall / sensitivity | TP / (TP + FN) | Actual positives |
| Precision | TP / (TP + FP) | Predicted positives |
| F1 | 2 × precision × recall / (precision + recall) | Harmonic combination of precision and recall |

F1 does not include true negatives and does not represent all consequences of a decision.


### Activity 3: Read a confusion matrix


**Individually, then compare with a partner · 4 minutes.** A hypothetical classifier has TN=8, FP=2, FN=3, TP=7. Calculate its accuracy, recall, and precision. Deliver the three fractions and their decimal values.

**Debrief:** Which denominator would you use to answer “Of the people the model flagged, how many actually had the recorded condition?”


In [ ]:
# accuracy_example = ...
# recall_example = ...
# precision_example = ...


**Your response:**

…


### Evaluate logistic regression against a baseline

`DummyClassifier(strategy="most_frequent")` learns only the most common training label and always predicts it. This gives our model a useful minimum comparison. The short `metric_row` function avoids repeating the same five calculations for each model. `zero_division=0` assigns zero when no positive predictions make precision undefined; interpret that result together with recall and the confusion matrix.

ROC AUC evaluates how well probability scores rank positive records above negative records across thresholds (0.5 corresponds to chance-level ranking; 1 is perfect ranking). It is not accuracy and does not select a deployment threshold.


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, recall_score, precision_score, f1_score,
                             roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
                             classification_report, RocCurveDisplay)

def metric_row(name, actual, predicted, score):
    return {
        "Model": name, "Accuracy": accuracy_score(actual, predicted),
        "Recall": recall_score(actual, predicted, zero_division=0),
        "Precision": precision_score(actual, predicted, zero_division=0),
        "F1": f1_score(actual, predicted, zero_division=0),
        "ROC AUC": roc_auc_score(actual, score)
    }

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
validation_results = pd.DataFrame([
    metric_row("Majority baseline", y_val, baseline.predict(X_val), baseline.predict_proba(X_val)[:, 1]),
    metric_row("Logistic regression", y_val, val_prediction, val_probability)
]).set_index("Model")
display(validation_results.round(3))
ConfusionMatrixDisplay.from_predictions(y_val, val_prediction, labels=[0, 1],
                                        display_labels=["Absent (0)", "Present (1)"], cmap="Blues")
plt.title("Logistic regression: validation records")
plt.show()


### Optional activity: change the threshold

A threshold turns a probability score into a flag. For example, flagging every probability at or above 0.30 will flag at least as many records as a threshold of 0.50 on the same scores. Recall cannot decrease when the threshold falls, but precision may rise or fall in a finite sample. A threshold must be chosen using development data and a stated objective, not the test results.


### Activity 4: Trade missed cases for more flags


**Groups of three · 5 minutes · optional.** Use `val_probability` to predict class 1 whenever the score is at least 0.30. Print the confusion matrix with `labels=[0, 1]`. Compare its false negatives and false positives with the default matrix above. Deliver the two changes and a sentence about the trade-off.

**Debrief:** Is a lower threshold automatically a better decision rule?


In [ ]:
# lower_prediction = (... >= 0.30).astype(int)
# print(confusion_matrix(..., ..., labels=[0, 1]))


**Your response:**

…


## 6. Freeze the workflow and evaluate the test set

For the main demonstration, our final choice is the logistic-regression pipeline with its default class threshold. We now refit that fixed pipeline on all development records (`X_dev`, the training and validation portions together). Its preparation is relearned from those development records. Only then do we predict the untouched test rows.

We clone the pipeline to create an unfitted copy; the earlier training-only `model` stays available for the optional validation exercises. This is the final test evaluation for this workflow. Do not choose a new model or threshold in response to these results and then describe a rerun on the same test set as an independent test.


In [ ]:
from sklearn.base import clone

final_model = clone(model)
final_model.fit(X_dev, y_dev)
test_prediction = final_model.predict(X_test)
test_probability = final_model.predict_proba(X_test)[:, 1]
display(pd.DataFrame([
    metric_row("Frozen logistic regression", y_test, test_prediction, test_probability)
]).set_index("Model").round(3))
print(classification_report(y_test, test_prediction, labels=[0, 1],
                            target_names=["Absent", "Present"], zero_division=0))


The report's **support** is the number of actual records in each class. Its macro average weights classes equally; its weighted average weights them by support. A small test set makes every mistake consequential for the reported percentages. This single split is a teaching estimate, not evidence of hospital readiness or future-heart-attack prediction.

The workflow is now: verify the outcome and record meanings → apply documented cleaning rules → reserve evaluation data → explore training records → fit preparation and model together → evaluate mistakes and compare a baseline → freeze choices → test once.


### Activity 5: Exit ticket


**Individually · 2 minutes.** Write one sentence for each item: (1) what `target=1` means after our recoding, (2) why scaling is learned from training data, and (3) what a false negative means here. Deliver three sentences.

**Debrief:** Name one reason this notebook's test score is insufficient to use the model in a hospital.


**Your response:**

…


## Optional extension A: a wider view of the training data

The following plots extend the earlier EDA. They continue to use training records only and do not change our frozen main model.

A **pair plot** shows every pair of selected numerical features; diagonal panels show single-variable distributions. A **regression plot** overlays a fitted straight-line summary on a scatterplot. These lines describe associations within the sample, not the classifier's decision rule and not causal effects.


In [ ]:
pair_features = ["age", "cholesterol", "resting_blood_pressure",
                 "max_heart_rate_achieved", "st_depression"]
g = sns.pairplot(train_data[pair_features + ["target"]], hue="target", corner=True,
                 diag_kind="hist", palette=class_colors, height=1.8)
g.figure.suptitle("Training records: pairwise relationships", y=1.02)
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for column, ax in zip(pair_features[1:], axes.flat):
    for label in [0, 1]:
        sns.regplot(data=train_data[train_data["target"] == label], x="age", y=column,
                    ax=ax, color=class_colors[label], label=str(label), ci=None,
                    scatter_kws={"s": 15, "alpha": 0.5})
    ax.legend(title="Recorded disease")
plt.tight_layout()
plt.show()


### All categorical features

This grid preserves the count-plot view across all seven categorical inputs. Read each panel's categories and counts before comparing it to another panel. Unequal group sizes and small categories can make impressions from raw counts misleading; use the row-normalized tables practiced above for within-group shares.


In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(13, 16))
for column, ax in zip(cat_feats, axes.flat):
    sns.countplot(data=train_data, x=column, hue="target", palette=class_colors, ax=ax)
    ax.set_title(column.replace("_", " "))
    ax.tick_params(axis="x", rotation=20)
axes.flat[-1].set_visible(False)
plt.tight_layout()
plt.show()


## Optional extension B: compare more classifiers

Different classifiers learn different patterns. Logistic regression combines weighted inputs; nearest neighbors uses similar training records; a support-vector classifier seeks a separating boundary; a decision tree learns branching rules; a random forest combines trees. Boosting combines models sequentially. Naive Bayes and discriminant analysis impose distributional assumptions; a neural network learns layered transformations.

The comparison below retains a broad set of examples. It is an illustration, not a search for a proven winner. All models receive the same split and preparation, although different models may benefit from different choices. The training data are small, so rankings may change with the split. We retain visible warnings: they can reveal convergence or numerical problems.

`clone` gives each classifier its own unfitted copy of the preprocessing. Each pipeline is fitted once and saved in `fitted_models`; plotting reuses the saved fits. We evaluate validation data only. Do not reuse the already-seen main test score to guide this exploration.

Support-vector classifiers return a **decision score** through `decision_function`: higher scores favor class 1, but these numbers are not probabilities and need not lie between 0 and 1. ROC curves and AUC require a ranking score, so they can use these values directly. For the other classifiers, we use their class-1 probabilities as scores. The `if` below makes this choice explicitly, and `val_scores` saves each model's scores for the plots.


Nearest neighbors uses `algorithm="kd_tree"` to look up nearby training rows. This is an exact search method: the classifier still predicts from the same five nearest neighbors.


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC, NuSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.neural_network import MLPClassifier

classifiers = {
    "Logistic regression": LogisticRegression(max_iter=1000, random_state=0),
    "Nearest neighbors": KNeighborsClassifier(n_neighbors=5, algorithm="kd_tree"),
    "Support vectors": SVC(random_state=0),
    "Nu SVC": NuSVC(random_state=0),
    "Decision tree": DecisionTreeClassifier(random_state=0),
    "Random forest": RandomForestClassifier(random_state=0),
    "AdaBoost": AdaBoostClassifier(random_state=0),
    "Gradient boosting": GradientBoostingClassifier(random_state=0),
    "Naive Bayes": GaussianNB(),
    "Linear DA": LinearDiscriminantAnalysis(),
    "Quadratic DA": QuadraticDiscriminantAnalysis(reg_param=0.1),
    "Neural net": MLPClassifier(hidden_layer_sizes=(20,), max_iter=2000, random_state=0)
}

fitted_models = {}
val_scores = {}
comparison_rows = []
for name, classifier in classifiers.items():
    candidate = Pipeline([("preprocess", clone(preprocess)), ("classifier", classifier)])
    candidate.fit(X_train, y_train)
    fitted_models[name] = candidate
    if name in ["Support vectors", "Nu SVC"]:
        val_scores[name] = candidate.decision_function(X_val)
    else:
        val_scores[name] = candidate.predict_proba(X_val)[:, 1]
    comparison_rows.append(metric_row(name, y_val, candidate.predict(X_val),
                                      val_scores[name]))

comparison = pd.DataFrame(comparison_rows).set_index("Model")
display(comparison.sort_values("F1", ascending=False).round(3))


### ROC curves and confusion matrices answer different questions

An ROC curve plots recall (true-positive rate) against false-positive rate as the threshold varies. The dashed diagonal represents chance-level ranking. The confusion matrices below instead describe each model's **default class prediction**, a single decision rule. A model with higher AUC need not have fewer false negatives at that default rule.

The QDA example uses mild covariance regularization (`reg_param=0.1`) because sparse categories and a small sample can make covariance estimates unstable. The neural-network example uses a small hidden layer. These choices make the examples practical; they are not tuned optimal settings.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
for name, candidate in fitted_models.items():
    RocCurveDisplay.from_predictions(y_val, val_scores[name],
                                      name=name, ax=ax)
ax.plot([0, 1], [0, 1], linestyle="--", color="gray")
ax.set_title("Validation ROC curves")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(4, 3, figsize=(12, 14))
for (name, candidate), ax in zip(fitted_models.items(), axes.flat):
    ConfusionMatrixDisplay.from_predictions(y_val, candidate.predict(X_val), labels=[0, 1],
                                            colorbar=False, ax=ax, cmap="Blues")
    ax.set_title(name)
plt.tight_layout()
plt.show()


## Optional extension C: tune a hyperparameter

A **parameter** such as a fitted coefficient is learned during training. A **hyperparameter** is a setting chosen outside that fitting step. Logistic regression's `C` controls regularization: smaller `C` means a stronger penalty on large coefficients.

`RandomizedSearchCV` samples candidate settings. Five-fold cross-validation divides **the training portion** into five parts, fits on four, scores the held-out part, and rotates that role. The pipeline relearns preprocessing within each fold. The average F1 score guides the search; it is not a test-set result.

`classifier__C` means the `C` setting of the pipeline step named `classifier`. We search a logarithmic range so both small and large scales are represented. `best_estimator_` is the selected pipeline, already refitted on the full training portion. We then inspect its validation performance. This optional exploration does not alter the main test evaluation above; a new finalized choice would need fresh independent evaluation.


In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from scipy.stats import loguniform

folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
search = RandomizedSearchCV(
    clone(model), {"classifier__C": loguniform(0.01, 100)},
    n_iter=10, scoring="f1", cv=folds, random_state=0
)
search.fit(X_train, y_train)
print("Selected settings:", search.best_params_)
print("Mean training-fold validation F1:", round(search.best_score_, 3))
tuned_model = search.best_estimator_
display(pd.DataFrame([
    metric_row("Tuned logistic regression", y_val, tuned_model.predict(X_val),
               tuned_model.predict_proba(X_val)[:, 1])
]).set_index("Model").round(3))


## Optional extension D: keep a record with MLflow

When you compare models, record the settings, data split, metric, and result together. MLflow organizes this information into **experiments**, each containing **runs**. Logging makes a result easier to find; it does not by itself ensure reproducibility or validate the model.

This optional example logs the earlier logistic-regression validation result. It does not retrain a model or require an account. MLflow is a new library for this extension. To try it, uncomment and run the installation line below, restart the kernel, run the preceding lesson cells, and set `RUN_MLFLOW = True` in the tracking cell. Keep it `False` to skip tracking. The example stores a SQLite tracking database in your home folder, outside the course repository.

`set_experiment` creates or selects a named experiment. `start_run` opens one run; `log_params` records settings and `log_metrics` records numeric results. We save the actual experiment ID returned by MLflow so the search works on your computer rather than relying on another person's run ID.


In [ ]:
# Uncomment and run only if you want to try MLflow.
# %pip install mlflow


In [ ]:
RUN_MLFLOW = False

if RUN_MLFLOW:
    from pathlib import Path
    import mlflow

    tracking_folder = Path.home() / "econ525-mlflow"
    tracking_folder.mkdir(exist_ok=True)
    tracking_file = (tracking_folder / "tracking.db").as_posix()
    mlflow.set_tracking_uri("sqlite:///" + tracking_file)
    mlflow.set_registry_uri("sqlite:///" + tracking_file)
    experiment = mlflow.set_experiment("Heart disease classification")
    with mlflow.start_run(run_name="logistic-validation"):
        mlflow.log_params({"classifier": "LogisticRegression", "C": 1.0,
                           "split_seed": 0, "evaluation_split": "validation"})
        mlflow.log_metrics(validation_results.loc["Logistic regression"].to_dict())
        mlflow.set_tag("outcome", "1 = recorded disease presence after recoding")
    runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
    display(runs[["run_id", "metrics.Accuracy", "metrics.Recall", "metrics.F1"]])
else:
    print("MLflow example skipped. The classification lesson is complete without it.")


### References

- [UCI Heart Disease dataset and variable definitions](https://archive.ics.uci.edu/dataset/45/heart+disease). The local CSV is a recoded Cleveland derivative.
- [scikit-learn: pipelines and composite estimators](https://scikit-learn.org/stable/modules/compose.html), [data leakage](https://scikit-learn.org/stable/common_pitfalls.html), and [classification metrics](https://scikit-learn.org/stable/modules/model_evaluation.html).
- [scikit-learn: OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) and [RandomizedSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html).
- [MLflow tracking](https://mlflow.org/docs/latest/ml/tracking/).
